<h1 style="font-size:36px;">Data Preprocessing</h1>

This notebook provides a first look at the dataset, including:

- Exploration of the dataset structure, types, and contents
- Checks for missing, duplicate, or inconsistent values
- Correction of formatting issues (numeric formats, UTF-8 encoding for special characters)
- Preparation of the data for further processing and eventual database loading

# 📥Step 1: Import libraries & load data

In [1]:
# Imports
import pandas as pd
import numpy as np

In [2]:
# Load raw CSV datasets into DataFrames
customers = pd.read_csv("data/raw/Fecom Inc Customer List.csv", sep=';')
geo = pd.read_csv("data/raw/Fecom Inc Geolocations.csv", sep=';', low_memory=False) # low_memory=False: read the full file to ensure correct dtypes
order_items = pd.read_csv("data/raw/Fecom Inc Order Items.csv", sep=';')
orders = pd.read_csv("data/raw/Fecom Inc Orders.csv", sep=';')
payments = pd.read_csv("data/raw/Fecom Inc Order Payments.csv", sep=';')
products = pd.read_csv("data/raw/Fecom Inc Products.csv", sep=';')
sellers = pd.read_csv("data/raw/Fecom Inc Sellers List.csv", sep=';')
reviews = pd.read_csv("data/raw/Fecom_Inc_Order_Reviews_No_Emojis.csv", sep=';')

 # 🔍Step 2: Inspect data for errors and missing values

<h2 style="color:#1443a3;">🗒️File #1 – Fecom Inc Customer List.csv</h2>

### 1. Inspect Structure & Data Types

In [3]:
# Preview the first 10 rows of the customers DataFrame
customers.head(10)

,Customer_Trx_ID,Subscriber_ID,Subscribe_Date,First_Order_Date,Customer_Postal_Code,Customer_City,Customer_Country,Customer_Country_Code,Age,Gender
0,1e959e1f5920cba43823fa9f95673b83,9765e039028279fd2e60bb620a451526,2023-07-08,2023-07-09,FR-75005,Paris,France,FR,29,Male
1,9877437582f263da7d7e30a90c57b8bb,a75e134e7eb6f96e2b0c716ac2a82efb,2024-03-23,2024-04-11,PL-00-001,Warsaw,Poland,PL,38,Male
2,fa6fbbb2080646acae977bf2e44af98b,2fdac27295500e820e43910e9a0aa8d8,2023-05-12,2023-06-01,NL-1012,Amsterdam,Netherlands,NL,35,Female
3,a4c9ff14ae7620126461ca55f36a76ea,e9ab8fd8ea96c85be2714c7f573fb7cd,2023-04-16,2023-04-26,IT-00144,Rome,Italy,IT,62,Male
4,93d5e378ae2f72a07db704f1f6716a7e,6384e6a7b021717616f25625b4bb0bf9,2023-05-26,2023-06-27,NL-1011,Amsterdam,Netherlands,NL,19,Male
5,3d78789a9236efa5e2c7a1c03d426b0f,b829dae01a25d79a42927f8a23bed610,2024-08-03,2024-08-09,FR-75005,Paris,France,FR,72,Female
6,b371150a5f99910091a68eda2d4c6d9d,c504ed3cb6e5d23e4ea7aa1c65321b69,2024-04-28,2024-06-07,SE-10031,Stockholm,Sweden,SE,29,Female
7,c4264a7d8bd33eccbe46b5f978f9965d,262e0bc6e703f729f370889579d1f0b4,2023-05-19,2023-07-07,FR-75002,Paris,France,FR,45,Female
8,9002e12e0f64d56ee2ad88c8f4949205,d5fcc930cdd39ae8b724014560a4c62c,2023-07-02,2023-07-03,IT-98121,Messina,Italy,IT,38,Male
9,a96aebd402f8c5f3993d287efb9154a5,a41f2e9a8fa027a863c24fb5e262ef01,2024-06-13,2024-07-01,ES-8005,Barcelona,Spain,ES,39,Male


In [4]:
# Get the dimensions of the customers DataFrame
customers.shape

(102727, 10)

Dataset has 102 727 rows and 10 columns

In [5]:
# Show column info, data types, and non-null counts
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102727 entries, 0 to 102726
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   Customer_Trx_ID        99441 non-null   object
 1   Subscriber_ID          102727 non-null  object
 2   Subscribe_Date         102727 non-null  object
 3   First_Order_Date       99441 non-null   object
 4   Customer_Postal_Code   102727 non-null  object
 5   Customer_City          102727 non-null  object
 6   Customer_Country       102727 non-null  object
 7   Customer_Country_Code  102727 non-null  object
 8   Age                    102727 non-null  int64 
 9   Gender                 102727 non-null  object
dtypes: int64(1), object(9)
memory usage: 7.8+ MB


**Observations:**
- Missing `Customer_Trx_ID` and `First_Order_Date` should be checked 
- Most columns are objects, meaning pandas interpreted them as strings. `Age` is integer
- Date columns (`Subscribe_Date`, `First_Order_Date`) are also objects

**Note:**
Here and throughout the notebook, we are not converting any date-like columns from object to datetime in Python. PostgreSQL will handle the type casting automatically during data loading, as long as DATE/TIMESTAMP types are defined at the DDL stage.


### 2. Handle Missing Values

In [6]:
# Show total missing values by column
customers.isna().sum()

Customer_Trx_ID          3286
Subscriber_ID               0
Subscribe_Date              0
First_Order_Date         3286
Customer_Postal_Code        0
Customer_City               0
Customer_Country            0
Customer_Country_Code       0
Age                         0
Gender                      0
dtype: int64

⚠️ Missing values in Customer_Trx_ID are problematic because this column is used as the primary key and as a foreign key to link with the orders table, so **rows with missing IDs cannot be reliably joined**

Let's preview the first few rows where Customer_Trx_ID is missing.

In [7]:
# Preview first 10 rows with missing Customer_Trx_ID
customers[customers['Customer_Trx_ID'].isna()].head(10)

,Customer_Trx_ID,Subscriber_ID,Subscribe_Date,First_Order_Date,Customer_Postal_Code,Customer_City,Customer_Country,Customer_Country_Code,Age,Gender
99441,NaN,0e0c08e088ec486e8784aefba35e17bc,2022-11-13,NaN,DE-10178,Berlin,Germany,DE,34,Male
99442,NaN,3ef786f65f114a27880ecdde737c638f,2023-06-12,NaN,NL-3012,Rotterdam,Netherlands,NL,34,Female
99443,NaN,2999799cc1a846b08098320f5aa3d8ad,2023-02-01,NaN,DE-30159,Hanover,Germany,DE,70,Female
99444,NaN,ed86c09317f34e209d497d04e3f84504,2024-08-22,NaN,RS-11030,Belgrade,Serbia,RS,39,Female
99445,NaN,a76544aba7844c7db3b1ab9bdcd7954e,2023-10-22,NaN,FR-75002,Paris,France,FR,26,Female
99446,NaN,bc572431a6d441878e754eff8d41ea06,2024-08-11,NaN,DE-80333,Munich,Germany,DE,35,Female
99447,NaN,ff320754577540749aab2c8afd5f796f,2024-09-04,NaN,GB-W1,London,United Kingdom,GB,52,Male
99448,NaN,4cbbb5d16eba41b6a57473f1125ae9d3,2024-09-25,NaN,GB-SE1,London,United Kingdom,GB,42,Male
99449,NaN,2a91f948ac5b48f18cf97b5eb3e324f8,2024-08-20,NaN,DE-10179,Berlin,Germany,DE,56,Male
99450,NaN,1c4301b1127f411b932df833b4dda5cc,2022-10-25,NaN,FR-75015,Paris,France,FR,41,Male


**Observation:**
- Missing Customer_Trx_ID values correspond to customers without any orders.
- This may be due to recent sign-ups, database errors, or because the customer never placed an order.

Let's dig deeper to see the share of missing Customer_Trx_ID that belong to recent subscribers without orders

In [8]:
# Temporarily convert subscription date to datetime for filtering
subscribe_dates = pd.to_datetime(customers['Subscribe_Date'])

# Filter rows with missing Customer_Trx_ID
missing_ids = customers[customers['Customer_Trx_ID'].isna()]

# Define cutoff date
cutoff_date = pd.Timestamp('2024-06-01')
cutoff_str = cutoff_date.strftime('%b %Y')

# Split all subscribers into older and recent
older_subs = customers[subscribe_dates <= cutoff_date]
recent_subs = customers[subscribe_dates > cutoff_date]

# Split missing rows into older and recent
missing_before = missing_ids[subscribe_dates[missing_ids.index] <= cutoff_date]
missing_after = missing_ids[subscribe_dates[missing_ids.index] > cutoff_date]

# Count and percentages relative to subscribers in each group
missing_before_count = len(missing_before)
missing_before_pct = missing_before_count / len(older_subs) * 100
missing_after_count = len(missing_after)
missing_after_pct = missing_after_count / len(recent_subs) * 100

# Print results
print(f"Missing Customer_Trx_ID for older subscribers (up to {cutoff_str}): {missing_before_count} "
      f"({missing_before_pct:.1f}% of subscribers up to cutoff)")
print(f"Missing Customer_Trx_ID for recent subscribers (after {cutoff_str}): {missing_after_count} "
      f"({missing_after_pct:.1f}% of subscribers after cutoff)")

Missing Customer_Trx_ID for older subscribers (up to Jun 2024): 1021 (1.2% of subscribers up to cutoff)
Missing Customer_Trx_ID for recent subscribers (after Jun 2024): 2265 (15.3% of subscribers after cutoff)


**Note:** The share of missing Customer_Trx_ID among older subscribers is very low (1.2%), which is unusually small for a typical customer dataset. Meanwhile, the recent subscribers show a much higher proportion (15.3%) of missing IDs. This suggests possible data inconsistencies or system-specific rules affecting how Customer_Trx_ID is recorded.

**Decision:** For simplicity, we will remove rows with missing Customer_Trx_ID. These are mostly recent subscribers without orders and a very small share of older subscribers. Since Customer_Trx_ID is a primary key needed to link with orders, keeping these rows would complicate analysis. Based on the observed inconsistencies noted above, we are primarily interested in active customers, so focusing on them by removing these rows is reasonable. These rows represent a small portion of the dataset, so this action won’t significantly affect order-related analysis.

**Alternative:** A separate table for subscribers without orders could be created, or a surrogate key could be assigned, but for now these rows are not needed and will be removed for simplicity.

In [9]:
# Drop rows with missing Customer_Trx_ID to get active customers
active_customers = customers.dropna(subset='Customer_Trx_ID')

In [10]:
# Verify NaNs remain after filtering out inactive customers
active_customers.isna().sum()

Customer_Trx_ID          0
Subscriber_ID            0
Subscribe_Date           0
First_Order_Date         0
Customer_Postal_Code     0
Customer_City            0
Customer_Country         0
Customer_Country_Code    0
Age                      0
Gender                   0
dtype: int64

All `Customer_Trx_ID` values are now present

### 3. Check Duplicates 

In [11]:
# Inspect number of duplicate records
active_customers.duplicated().sum()

np.int64(0)

No duplicates detected

Finally, we will check the uniqueness of the column `Customer_Trx_ID` that serves as the primary key to ensure that each record is uniquely identifiable and suitable for relational integrity

In [12]:
# Count duplicate primary keys
active_customers['Customer_Trx_ID'].duplicated().sum()

np.int64(0)

No duplicates detected, confirming all records are uniquely identifiable

✅**Dataset is ready for further processing**

### 4. Save Preprocessed Data

In [13]:
# Save the preprocessed DataFrame to CSV
active_customers.to_csv("data/processed/Fecom Inc Customer List.csv", index=False, sep=';')

 <h2 style="color:#1443a3;">🗒️File #2 – Fecom Inc Geolocations.csv</h2>

### 1. Inspect Structure & Data Types 

In [14]:
# Preview the first 10 rows of the geo DataFrame
geo.head(10)

,Geo_Postal_Code,Geo_Lat,Geo_Lon,Geolocation_City,Geo_Country
0,NL-5211,"51,7000","5,3167",'s-Hertogenbosch,Netherlands
1,NL-5212,"51,7000","5,3167",'s-Hertogenbosch,Netherlands
2,NL-5213,"51,7000","5,3167",'s-Hertogenbosch,Netherlands
3,NL-5214,"51,7000","5,3167",'s-Hertogenbosch,Netherlands
4,NL-5215,"51,7000","5,3167",'s-Hertogenbosch,Netherlands
5,ES-15001,"43,3667","-8,3833",A Coruña,Spain
6,ES-15002,"43,3667","-8,3833",A Coruña,Spain
7,ES-15003,"43,3667","-8,3833",A Coruña,Spain
8,ES-15004,"43,3667","-8,3833",A Coruña,Spain
9,ES-15005,"43,3667","-8,3833",A Coruña,Spain


In [15]:
# Get the dimensions of the DataFrame
geo.shape

(1000163, 5)

Dataset has 1 000 163 rows and 5 columns

In [16]:
# Show column info, data types, and non-null counts
geo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Geo_Postal_Code   1685 non-null   object
 1   Geo_Lat           1685 non-null   object
 2   Geo_Lon           1685 non-null   object
 3   Geolocation_City  1685 non-null   object
 4   Geo_Country       1685 non-null   object
dtypes: object(5)
memory usage: 38.2+ MB


**Observations:**
- All columns are objects
- Only 1,685 non-null rows out of over 1 million 
- A large portion of the dataset contains missing values

### 2. Handle Missing Values

In [17]:
# Show total missing values by column
geo.isna().sum()

Geo_Postal_Code     998478
Geo_Lat             998478
Geo_Lon             998478
Geolocation_City    998478
Geo_Country         998478
dtype: int64

⚠️Almost all rows are missing values in every column (998 478 out of 1 000 163), showing that the dataset contains mostly empty rows

We will remove rows where all values are missing, as they don’t contain any useful information.

In [18]:
# Remove rows where all values are NaN
geo = geo.dropna(how='all')

In [19]:
# Verify NaNs remain 
geo.isna().sum()

Geo_Postal_Code     0
Geo_Lat             0
Geo_Lon             0
Geolocation_City    0
Geo_Country         0
dtype: int64

### 3. Check Duplicates 

In [20]:
# Inspect number of duplicate records
geo.duplicated().sum()

np.int64(0)

No duplicates detected

`Geo_Postal_Code` and `Geolocation_City` should form a unique pair, since each payment within an order has its own sequence number. Let`s check for duplicates to ensure this rule holds

In [21]:
# Checking for repeated Geo_Postal_Code and Geolocation_City pairs
geo.duplicated(subset=['Geo_Postal_Code','Geolocation_City']).sum()

np.int64(0)

The result is zero, confirming that the combination is unique and the data is consistent

### 4. Correct Formatting Issues

The original file had a few issues that would cause import problems:
 1. Decimal numbers used commas instead of dots (e.g., 51,7000) in the following columns: Geo_Lat, Geo_Lon. PostgreSQL expects a dot as the decimal separator
 2. Special characters in city names (e.g., A Coruña) require UTF-8 encoding during import

To fix these issues:
 - Replace commas with dots in numeric columns
 - Verify that the file uses the correct UTF-8 encoding

In [22]:
# Preparing the geolocation CSV for PostgreSQL import

In [23]:
# Replace commas with dots in numeric columns
geo['Geo_Lat'] = geo['Geo_Lat'].str.replace(',', '.').astype(float)
geo['Geo_Lon'] = geo['Geo_Lon'].str.replace(',', '.').astype(float)

# Save cleaned file
geo.to_csv("data/processed/Fecom Inc Geolocations.csv", index=False, sep=';', encoding='utf-8')

✅**Dataset is ready for further processing**

### 5. Save Preprocessed Data

In [24]:
# Save the preprocessed DataFrame to CSV
geo.to_csv("data/processed/Fecom Inc Geolocations.csv", index=False, sep=';')

<h2 style="color:#1443a3;"> 🗒️File #3 – Fecom Inc Order Items.csv </h2>

### 1. Inspect Structure & Data Types

In [25]:
# Preview the first 10 rows of the order_items DataFrame
order_items.head(10) 

,Order_ID,Order_Item_ID,Product_ID,Seller_ID,Shipping_Limit_Date,Price,Freight_Value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2023-09-19 09:45,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2023-05-03 11:05,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2024-01-18 14:48,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2024-08-15 10:10,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2023-02-13 13:57,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2023-05-23 03:55,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2023-12-14 12:10,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2024-07-10 12:30,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2024-03-26 18:31,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2024-07-06 14:10,53.99,11.40


In [26]:
# Show column info, data types, and non-null counts
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Order_ID             112650 non-null  object 
 1   Order_Item_ID        112650 non-null  int64  
 2   Product_ID           112650 non-null  object 
 3   Seller_ID            112650 non-null  object 
 4   Shipping_Limit_Date  112650 non-null  object 
 5   Price                112650 non-null  float64
 6   Freight_Value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


**Observations:**
- DataFrame has 112 650 rows and 7 columns
- Numeric columns: Price and Freight_Value (float64), Order_Item_ID (int64).
- Object columns: Order_ID, Product_ID, Seller_ID, Shipping_Limit_Date.
- Shipping_Limit_Date is currently an object

**Note:**
We are not converting any date-like columns from object to datetime in Python. PostgreSQL will handle the type casting during data loading once DATE/TIMESTAMP types are defined at the DDL stage.

In [27]:
#Unique values per column
order_items.nunique()

Order_ID               98666
Order_Item_ID             21
Product_ID             32951
Seller_ID               3095
Shipping_Limit_Date    54615
Price                   5968
Freight_Value           6999
dtype: int64

- Shipping_Limit_Date and Product_ID have high cardinality
- Order_Item_ID currently shows the position of a product within each order (1–21). It will be renamed to Item_Position in PostgreSQL before further analysis
- Most columns show reasonable variability for analysis

### 2. Handle Missing Values

In [28]:
# Show total missing values by column
order_items.isna().sum()

Order_ID               0
Order_Item_ID          0
Product_ID             0
Seller_ID              0
Shipping_Limit_Date    0
Price                  0
Freight_Value          0
dtype: int64

All columns have no missing values.
No action required for missing values

### 3. Check Duplicates 

In [29]:
# Inspect number of duplicate records
order_items.duplicated().sum()

np.int64(0)

No duplicates detected

`Order_ID` and `Order_Item_ID` should form a unique pair, since each payment within an order has its own sequence number. Let`s check for duplicates to ensure this rule holds

In [30]:
# Checking for repeated Order_ID + Order_Item_ID pairs
order_items.duplicated(subset=['Order_ID', 'Order_Item_ID']).sum()

np.int64(0)

The result is zero, confirming that the combination is unique and the data is consistent

✅**Dataset is ready for further processing**

### 4. Save Preprocessed Data

In [31]:
# Save the preprocessed DataFrame to CSV
order_items.to_csv("data/processed/Fecom Inc Order Items.csv", index=False, sep=';')

File can be loaded as is, no changes needed before import.

<h2 style="color:#1443a3;"> 🗒️File #4 – Fecom_Inc_Order_Reviews_No_Emojis.csv </h2>

### 1. Inspect Structure & Data Types

In [32]:
# Preview the first 10 rows of the reviews DataFrame
reviews.head(10) 

,Review_ID,Order_ID,Review_Score,Review_Comment_Title_En,Review_Comment_Message_En,Review_Creation_Date,Review_Answer_Timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2024-01-18 00:00,2024-01-18 21:46
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2024-03-10 00:00,2024-03-11 03:05
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2024-02-17 00:00,2024-02-18 14:36
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,I received it well before the stipulated deadl...,2023-04-21 00:00,2023-04-21 22:02
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,"Congratulations lannister stores, I loved shop...",2024-03-01 00:00,2024-03-02 10:26
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,NaN,NaN,2024-04-13 00:00,2024-04-16 00:39
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,NaN,NaN,2023-07-16 00:00,2023-07-18 19:30
7,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,NaN,NaN,2024-08-14 00:00,2024-08-14 21:36
8,a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,NaN,NaN,2023-05-17 00:00,2023-05-18 12:05
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recommend,efficient device. On the website the brand of ...,2024-05-22 00:00,2024-05-23 16:45


In [33]:
# Show column info, data types, and non-null counts
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99223 entries, 0 to 99222
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Review_ID                  99223 non-null  object
 1   Order_ID                   99223 non-null  object
 2   Review_Score               99223 non-null  int64 
 3   Review_Comment_Title_En    11541 non-null  object
 4   Review_Comment_Message_En  40876 non-null  object
 5   Review_Creation_Date       99223 non-null  object
 6   Review_Answer_Timestamp    99223 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


DataFrame has 99 223 rows and 7 columns

Data Types:
- Review_Score: int64
- IDs and dates: object
- Comment columns: object

**Note:**
We are not converting any date-like columns from object to datetime in Python. PostgreSQL will handle the type casting during data loading once DATE/TIMESTAMP types are defined at the DDL stage.

### 2. Handle Missing Values

In [34]:
# Show total missing values by column
reviews.isna().sum()

Review_ID                        0
Order_ID                         0
Review_Score                     0
Review_Comment_Title_En      87682
Review_Comment_Message_En    58347
Review_Creation_Date             0
Review_Answer_Timestamp          0
dtype: int64

Missing values in Review_Comment_Title_En (87 682) and Review_Comment_Message_En (58 347) are acceptable, because not all reviews include a title or message.
Other columns have no missing values

### 3. Check Duplicates 

In [35]:
# Inspect number of duplicate records
reviews.Order_ID.duplicated().sum()

np.int64(551)

Some orders have multiple reviews. This is plausible if customers update or add multiple reviews for the same order

In [36]:
reviews.Review_ID.duplicated().sum()

np.int64(814)

`Review_ID` has 814 duplicates. IDs are not strictly unique, likely due to system errors or multiple entries per review


In [37]:
reviews.duplicated().sum()

np.int64(0)

 No full-row duplicates detected 

✅**Dataset is ready for further processing**

### 4. Save Preprocessed Data

In [38]:
# Save the preprocessed DataFrame to CSV
reviews.to_csv("data/processed/Fecom_Inc_Order_Reviews_No_Emojis.csv", index=False, sep=';')

File can be loaded as is, no changes needed before import.

<h2 style="color:#1443a3;"> 🗒️File #5 – Fecom Inc Orders.csv </h2>

### 1. Inspect Structure & Data Types

In [39]:
# Preview the first 10 rows of the orders DataFrame
orders.head(10) 

,Order_ID,Customer_Trx_ID,Order_Status,Order_Purchase_Timestamp,Order_Approved_At,Order_Delivered_Carrier_Date,Order_Delivered_Customer_Date,Order_Estimated_Delivery_Date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2023-10-02 10:56,2023-10-02 11:07,2023-10-04 19:55,2023-10-10 21:25,2023-10-18 00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2024-07-24 20:41,2024-07-26 03:24,2024-07-26 14:31,2024-08-07 15:27,2024-08-13 00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2024-08-08 08:38,2024-08-08 08:55,2024-08-08 13:50,2024-08-17 18:06,2024-09-04 00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2023-11-18 19:28,2023-11-18 19:45,2023-11-22 13:39,2023-12-02 00:28,2023-12-15 00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2024-02-13 21:18,2024-02-13 22:20,2024-02-14 19:46,2024-02-16 18:17,2024-02-26 00:00
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2023-07-09 21:57,2023-07-09 22:10,2023-07-11 14:58,2023-07-26 10:57,2023-08-01 00:00
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2023-04-11 12:22,2023-04-13 13:25,NaN,NaN,2023-05-09 00:00
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2023-05-16 13:10,2023-05-16 13:22,2023-05-22 10:07,2023-05-26 12:55,2023-06-07 00:00
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2023-01-23 18:29,2023-01-25 02:50,2023-01-26 14:16,2023-02-02 14:08,2023-03-06 00:00
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2023-07-29 11:55,2023-07-29 12:05,2023-08-10 19:45,2023-08-16 17:14,2023-08-23 00:00


In [40]:
orders.Order_ID.duplicated().sum()

np.int64(0)

In [41]:
# Show column info, data types, and non-null counts
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Order_ID                       99441 non-null  object
 1   Customer_Trx_ID                99441 non-null  object
 2   Order_Status                   99441 non-null  object
 3   Order_Purchase_Timestamp       99441 non-null  object
 4   Order_Approved_At              99281 non-null  object
 5   Order_Delivered_Carrier_Date   97658 non-null  object
 6   Order_Delivered_Customer_Date  96476 non-null  object
 7   Order_Estimated_Delivery_Date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


- `orders DataFrame` has 99,441 rows and 8 columns
- All columns are objects

**Note:**
We are not converting any date-like columns from object to datetime in Python. PostgreSQL will handle the type casting during data loading once DATE/TIMESTAMP types are defined at the DDL stage.

### 2. Handle Missing Values

In [42]:
# Show total missing values by column
orders.isna().sum()

Order_ID                            0
Customer_Trx_ID                     0
Order_Status                        0
Order_Purchase_Timestamp            0
Order_Approved_At                 160
Order_Delivered_Carrier_Date     1783
Order_Delivered_Customer_Date    2965
Order_Estimated_Delivery_Date       0
dtype: int64

Missing values are in Order_Approved_At, Order_Delivered_Carrier_Date, and Order_Delivered_Customer_Date.
Most likely reason: orders are pending, canceled, or not yet delivered

**Decision:** Keep the missing values as is, since they are useful for analysis of pending or undelivered orders.

### 3. Check Duplicates 

In [43]:
# Inspect number of duplicate records
orders.duplicated().sum()

np.int64(0)

No duplicates detected

Finally, we will check the uniqueness of `Order_ID`, which serves as the primary key, and also verify that `Customer_Trx_ID` is unique, to ensure each record is properly identifiable and maintains relational integrity

In [44]:
# Count duplicate primary keys
orders['Order_ID'].duplicated().sum()

np.int64(0)

In [45]:
# Verify that Customer_Trx_ID is unique
orders['Customer_Trx_ID'].duplicated().sum()

np.int64(0)

No duplicates detected, confirming all records are uniquely identifiable

✅**Dataset is ready for further processing**

### 4. Save Preprocessed Data

In [46]:
# Save the preprocessed DataFrame to CSV
orders.to_csv("data/processed/Fecom Inc Orders.csv", index=False, sep=';')

File can be loaded as is, no changes needed before import.

<h2 style="color:#1443a3;"> 🗒️File #6 – Fecom Inc Order Payments.csv </h2>

### 1. Inspect Structure & Data Types

In [47]:
# Preview the first 10 rows of the payments DataFrame
payments.head(10)

,Order_ID,Payment_Sequential,Payment_Type,Payment_Installments,Payment_Value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
5,298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
6,771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
7,3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
8,1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
9,0573b5e23cbd798006520e1d5b4c6714,1,debit_card,1,51.95


In [48]:
orders.Customer_Trx_ID.value_counts().nunique()

1

In [49]:
payments.nunique()

Order_ID                99440
Payment_Sequential         29
Payment_Type                5
Payment_Installments       23
Payment_Value           29077
dtype: int64

In [50]:
# Show column info, data types, and non-null counts
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Order_ID              103886 non-null  object 
 1   Payment_Sequential    103886 non-null  int64  
 2   Payment_Type          103886 non-null  object 
 3   Payment_Installments  103886 non-null  int64  
 4   Payment_Value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


**Observations:**
- DataFrame has 103,886 rows and 5 columns
- All columns have no missing values.
- Data types: 2 integers (Payment_Sequential, Payment_Installments), 1 float (Payment_Value), 2 objects (Order_ID, Payment_Type)

### 2. Handle Missing Values

In [51]:
# Show total missing values by column
payments.isna().sum()

Order_ID                0
Payment_Sequential      0
Payment_Type            0
Payment_Installments    0
Payment_Value           0
dtype: int64

### 3. Check Duplicates 

In [52]:
# Inspect number of duplicate records
payments.duplicated().sum()

np.int64(0)

No duplicates detected

`Order_ID` and `Payment_Sequential` should form a unique pair, since each payment within an order has its own sequence number. Let`s check for duplicates to ensure this rule holds

In [53]:
# Checking for repeated Order_ID + Payment_Sequential pairs
payments.duplicated(subset=['Order_ID', 'Payment_Sequential']).sum()

np.int64(0)

The result is zero, confirming that the combination is unique and the data is consistent

✅**Dataset is ready for further processing**

### 4. Save Preprocessed Data

In [54]:
# Save the preprocessed DataFrame to CSV
reviews.to_csv("data/processed/Fecom_Inc_Order_Reviews_No_Emojis.csv", index=False, sep=';')

File can be loaded as is, no changes needed before import.

<h2 style="color:#1443a3;"> 🗒️File #7 – Fecom Inc Products.csv </h2>

### 1. Inspect Structure & Data Types

In [55]:
# Preview the first 10 rows of the products DataFrame
products.head(10) 

,Product_ID,Product_Category_Name,Product_Weight_Gr,Product_Length_Cm,Product_Height_Cm,Product_Width_Cm
0,1e9e8ef04dbcff4541ed26657ea517e5,Perfumery,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,Art,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,Sports_Leisure,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,Baby,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,Housewares,625.0,20.0,17.0,13.0
5,41d3672d4792049fa1779bb35283ed13,Musical_Instruments,200.0,38.0,5.0,11.0
6,732bd381ad09e530fe0a5f457d81becb,Cool_Stuff,18350.0,70.0,24.0,44.0
7,2548af3e6e77a690cf3eb6368e9ab61e,Furniture_Decor,900.0,40.0,8.0,40.0
8,37cc742be07708b53a98702e77a21a02,Home_Appliances,400.0,27.0,13.0,17.0
9,8c92109888e8cdf9d66dc7e463025574,Toys,600.0,17.0,10.0,12.0


In [56]:
# Show column info, data types, and non-null counts
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Product_ID             32951 non-null  object 
 1   Product_Category_Name  32328 non-null  object 
 2   Product_Weight_Gr      32949 non-null  float64
 3   Product_Length_Cm      32949 non-null  float64
 4   Product_Height_Cm      32949 non-null  float64
 5   Product_Width_Cm       32949 non-null  float64
dtypes: float64(4), object(2)
memory usage: 1.5+ MB


**Observations:**
- `products` DataFrame has 32,951 rows and 6 columns
- Data types: 4 floats (numeric), 2 objects

### 2. Handle Missing Values

In [57]:
# Show total missing values by column
products.isna().sum()

Product_ID                 0
Product_Category_Name    623
Product_Weight_Gr          2
Product_Length_Cm          2
Product_Height_Cm          2
Product_Width_Cm           2
dtype: int64

- Product_ID and Product_Weight_Gr have almost no missing values
- Product_Category_Name has 623 missing values, likely because some products are not yet categorized
- Dimensions (Length, Height, Width) and weight are mostly complete (2 missing)


Missing values are left as is and can be addressed later during analysis if needed.

In [58]:
# Select products with missing weight to inspect incomplete records
products[products['Product_Weight_Gr'].isna()]

,Product_ID,Product_Category_Name,Product_Weight_Gr,Product_Length_Cm,Product_Height_Cm,Product_Width_Cm
8578,09ff539a621711667c43eba6a3bd8466,Baby,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN


### 3. Check Duplicates 

In [59]:
# Inspect number of duplicate records
products.duplicated().sum()

np.int64(0)

No duplicates detected

Finally, we will check the uniqueness of the column `Product_ID` that serves as the primary key to ensure that each record is uniquely identifiable and suitable for relational integrity

In [60]:
# Count duplicate primary keys
products['Product_ID'].duplicated().sum()

np.int64(0)

No duplicates detected, confirming all records are uniquely identifiable

✅**Dataset is ready for further processing**

### 4. Save Preprocessed Data

In [61]:
# Save the preprocessed DataFrame to CSV
reviews.to_csv("data/processed/Fecom_Inc_Order_Reviews_No_Emojis.csv", index=False, sep=';')

File can be loaded as is, no changes needed before import.

<h2 style="color:#1443a3;"> 🗒️File #8 – Fecom Inc Sellers List.csv </h2>

### 1. Inspect Structure & Data Types

In [62]:
# Preview the first 10 rows of the sellers DataFrame
sellers.head(10) 

,Seller_ID,Seller_Name,Seller_Postal_Code,Seller_City,Country_Code,Seller_Country
0,d1b65fc7debc3361ea86b5f14c68d2e2,NeuroLabsX,DE-14469,Potsdam,DE,Germany
1,51a04a8a6bdcb23deccc82b0b80742cf,SwiftLabs,DE-6108,Halle (Saale),DE,Germany
2,e49c26c3edfa46d227d5121a6b6e4d37,EcoFutures,ES-33003,Oviedo,ES,Spain
3,1b938a7ec6ac5061a66a3766e0e75f90,HyperHub,DE-6112,Halle (Saale),DE,Germany
4,a7a9b880c49781da66651ccf4ba9ac38,EliteAI,DE-18069,Rostock,DE,Germany
5,7b8e8ec35bad4b0ef7e3963650b0a87b,InnovaGlobal,CH-6501,Bellinzona,CH,Switzerland
6,166e8f1381e09651983c38b1f6f91c11,HorizonHub,CH-3603,Thun,CH,Switzerland
7,4cf490a58259286ada5ba8525ba9e84a,HyperLogistics,DE-47802,Krefeld,DE,Germany
8,2ff97219cb8622eaf3cd89b7d9c09824,ElevateEnterprises,DE-34117,Kassel,DE,Germany
9,8bd0e3abda539b9479c4b44a691be1ec,VisionaryLabs,BE-3603,Genk,BE,Belgium


In [63]:
# Show column info, data types, and non-null counts
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Seller_ID           3095 non-null   object
 1   Seller_Name         3095 non-null   object
 2   Seller_Postal_Code  3095 non-null   object
 3   Seller_City         3095 non-null   object
 4   Country_Code        3095 non-null   object
 5   Seller_Country      3095 non-null   object
dtypes: object(6)
memory usage: 145.2+ KB


**Observations:**
 - `sellers` DataFrame has 3,095 rows and 6 columns.
- All columns are objects

### 2. Handle Missing Values


In [64]:
# Show total missing values by column
sellers.isna().sum()

Seller_ID             0
Seller_Name           0
Seller_Postal_Code    0
Seller_City           0
Country_Code          0
Seller_Country        0
dtype: int64

All columns have no missing values

### 3. Check Duplicates 

In [65]:
# Inspect number of duplicate records
sellers.duplicated().sum()

np.int64(0)

No duplicates detected

Finally, we will check the uniqueness of the column `Seller_ID` that serves as the primary key to ensure that each record is uniquely identifiable and suitable for relational integrity

In [66]:
# Count duplicate primary keys
sellers['Seller_ID'].duplicated().sum()

np.int64(0)

No duplicates detected, confirming all records are uniquely identifiable

✅**Dataset is ready for further processing**

### 4. Save Preprocessed Data

In [67]:
# Save the preprocessed DataFrame to CSV
reviews.to_csv("data/processed/Fecom_Inc_Order_Reviews_No_Emojis.csv", index=False, sep=';')

File can be loaded as is, no changes needed before import.

# 📝 Summary of Data Preprocessing

In this notebook, we performed an initial exploration and cleaning of the Fecom Inc datasets, including Customers, Geolocations, Order Items, Orders, Payments, Products, Sellers, and Reviews. The following steps were completed:
### 1. Data Inspection
- Examined dataset structure, column types, and sample rows
- Identified missing, duplicate, or inconsistent values
- Checked primary and unique keys for relational integrity:
  - `Customer_Trx_ID`
  - `Order_ID`
  - `Order_Item_ID`
  - `Payment_Sequential`
  - `Geo_Postal_Code` + `Geolocation_City`
### 2. Handling Missing Values
- Removed rows with missing primary keys (Customer_Trx_ID), as these mostly corresponded to subscribers without orders and likely represent data inconsistencies rather than actual missing information
- Removed geolocation rows where all columns were missing
- Verified that other missing values (e.g., review titles/messages) were acceptable
### 3. Duplicate Checks
- Confirmed no duplicate records across all datasets
- Ensured logical unique combinations:
  - `Order_ID` + `Order_Item_ID`
  - `Geo_Postal_Code` + `Geolocation_City`
  - `Order_ID` + `Payment_Sequential`
### 4. Data Cleaning & Formatting
- Corrected decimal separators in numeric columns (`Geo_Lat`, `Geo_Lon`)
- Ensured UTF-8 encoding for special characters
- Kept date columns as objects, since PostgreSQL will handle type conversion during import
### 5. Data Export
- Saved cleaned and preprocessed datasets to `data/processed/` directory in CSV format, ready for database loading and further analysis

**Database structure note:** Normalization should be considered to reduce redundancy, improve data consistency, and simplify queries. For example, instead of storing city and country names repeatedly in the customers table, a `geolocation_id` could link to a separate geolocations table. This approach would make updates easier, ensure consistent data, and support more efficient queries 

## ✅The datasets are now consistent, free of duplicates, and ready for import into a relational database